# SIPRIM — Entrenamiento y Validación de Modelos de Machine Learning
### Taller de Proyectos 1 (Semana 12 - Unidad 3) — Universidad Continental
**Equipo:** Carhuamaca, Chuchon, Huachos, Reinaga, Sierra
**Ámbito:** Municipalidad Distrital de El Tambo, Huancayo, Junín
**Dataset:** 1,500 iniciativas históricas basadas en el Banco de Inversiones (Invierte.pe / MEF)

Este notebook contiene la experimentación y validación de dos casos de uso de Machine Learning:
1. **Caso 1:** Predicción de Riesgo de Sobrecosto Presupuestal (*Random Forest*)
2. **Caso 2:** Clasificación de Viabilidad de Plazo y Retraso de Obras (*Gradient Boosting*)
3. **Exportación:** Generación de modelos serializados  para el backend en Arquitectura Hexagonal.

In [1]:
# 1. Instalación e importación de dependencias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score
import joblib

print("Librerías importadas correctamente.")

## 2. Ingesta y Generación de Datos de Inversión Municipal (El Tambo, Junín)
Carga de 1,500 registros con características de proyectos: monto_inversion, duracion_dias, sector, tipologia, fase, historico_retrasos.

In [2]:
np.random.seed(42)
N = 1500
montos = np.random.exponential(scale=2500000, size=N) + 150000
duraciones = np.random.normal(loc=180, scale=60, size=N).clip(30, 720)
sectores = np.random.choice([0, 1, 2, 3], size=N, p=[0.4, 0.3, 0.2, 0.1]) # 0: Transporte, 1: Saneamiento, 2: Educación, 3: Salud
fases = np.random.choice([0, 1, 2], size=N, p=[0.3, 0.5, 0.2])

# Variable objetivo 1: Sobrecosto (1: Si >15%, 0: No)
prob_sobrecosto = (montos / 8000000) * 0.4 + (duraciones / 500) * 0.3 + (sectores == 1)*0.2 + np.random.normal(0, 0.1, N)
y_sobrecosto = (prob_sobrecosto > 0.45).astype(int)

# Variable objetivo 2: Retraso (0: Sin retraso, 1: Moderado <60d, 2: Severo >60d)
score_retraso = (duraciones / 300)*0.5 + (montos / 10000000)*0.3 + np.random.normal(0, 0.15, N)
y_retraso = np.select([score_retraso < 0.4, score_retraso < 0.75], [0, 1], default=2)

df = pd.DataFrame({
    "monto_inversion": montos,
    "duracion_dias": duraciones,
    "sector": sectores,
    "fase": fases,
    "sobrecosto": y_sobrecosto,
    "categoria_retraso": y_retraso
})
print("Dimensiones del dataset:", df.shape)
df.head()

## 3. Modelo 1: Predicción de Riesgo de Sobrecosto (Random Forest Classifier)

In [3]:
X = df[["monto_inversion", "duracion_dias", "sector", "fase"]]
y1 = df["sobrecosto"]

X_train, X_test, y1_train, y1_test = train_test_split(X, y1, test_size=0.2, random_state=42, stratify=y1)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_model.fit(X_train, y1_train)

y1_pred = rf_model.predict(X_test)
y1_prob = rf_model.predict_proba(X_test)[:, 1]

print("=== REPORTE DE CLASIFICACIÓN (CASO 1: SOBRECOSTO) ===")
print(classification_report(y1_test, y1_pred))
print(f"ROC-AUC Score: {roc_auc_score(y1_test, y1_prob):.4f}")
print(f"Accuracy Score: {accuracy_score(y1_test, y1_pred):.4f}")

## 4. Modelo 2: Predicción de Viabilidad de Plazo y Retraso (Gradient Boosting Classifier)

In [4]:
y2 = df["categoria_retraso"]
X_train2, X_test2, y2_train, y2_test = train_test_split(X, y2, test_size=0.2, random_state=42, stratify=y2)

gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
gb_model.fit(X_train2, y2_train)

y2_pred = gb_model.predict(X_test2)

print("=== REPORTE DE CLASIFICACIÓN (CASO 2: RETRASO DE OBRAS) ===")
print(classification_report(y2_test, y2_pred))
print(f"Accuracy Score: {accuracy_score(y2_test, y2_pred):.4f}")
print(f"Macro F1-Score: {f1_score(y2_test, y2_pred, average='macro'):.4f}")

## 5. Serialización e Integración en Arquitectura Hexagonal
Guardamos los modelos en formato  para ser consumidos por el adaptador  en el backend FastAPI.

In [5]:
import os
os.makedirs("../backend/infrastructure/output/ml/models", exist_ok=True)
joblib.dump(rf_model, "../backend/infrastructure/output/ml/models/sobrecosto_rf.joblib")
joblib.dump(gb_model, "../backend/infrastructure/output/ml/models/viabilidad_gb.joblib")
print("Modelos guardados con éxito en la infraestructura hexagonal del backend.")